# NVIDIA RAG Plugin for NeMo Agent Toolkit

This notebook demonstrates how to use the NVIDIA RAG plugin with NeMo Agent toolkit. The plugin integrates [NVIDIA RAG](https://github.com/NVIDIA-AI-Blueprints/rag) to provide document retrieval and AI-generated responses within your agent workflows.

**This tutorial uses Milvus Lite** - a lightweight, file-based vector database that requires no Docker or external services!

We will cover:
- Setting up the prerequisites (Milvus Lite, API keys)
- Installing the RAG plugin
- **Custom document ingestion** into a local Milvus Lite database
- Using the plugin via CLI (`nat run`)
- Using the plugin programmatically via Python API
- Using the NVIDIA RAG client directly without the workflow system

## Table of Contents

- [0) Setup](#setup)
  - [0.1) Prerequisites](#prereqs)
  - [0.2) API Keys](#api-keys)
  - [0.3) Vector Database (Milvus Lite)](#milvus-setup)
  - [0.4) Installing the RAG Plugin](#installing-plugin)
- [1) Custom Document Ingestion](#custom-ingestion)
  - [1.1) Create Sample Documents](#create-sample-docs)
  - [1.2) Generate Embeddings](#generate-embeddings)
  - [1.3) Create Collection and Ingest Documents](#ingest-documents)
  - [1.4) Verify Ingestion](#verify-ingestion)
- [2) Understanding the RAG Functions](#understanding-functions)
  - [2.1) nvidia_rag_query - Get AI-Generated Responses](#rag-query)
  - [2.2) nvidia_rag_search - Search Document Chunks](#rag-search)
- [3) Using the RAG Plugin via CLI](#cli-usage)
  - [3.1) Creating a Workflow Configuration](#cli-config)
  - [3.2) Running with nat run Command](#cli-run)
- [4) Using the RAG Plugin Programmatically](#python-api)
  - [4.1) Using run_workflow Function](#run-workflow)
  - [4.2) Using WorkflowBuilder Directly](#workflow-builder)
- [5) Using NVIDIA RAG Client Directly](#direct-rag)
  - [5.1) Without the Workflow System](#without-workflow)
- [6) Troubleshooting](#troubleshooting)

<span style="color:rgb(0, 31, 153); font-style: italic;">Note: In Google Colab use the Table of Contents tab to navigate.</span>

<a id="setup"></a>
## 0) Setup

<a id="prereqs"></a>
### 0.1) Prerequisites

- **Platform:** Linux or macOS (Milvus Lite is not supported on Windows)
- **Python:** version 3.11, 3.12, or 3.13
- **NeMo Agent Toolkit:** Installed and configured

> **Note:** This tutorial uses Milvus Lite, which stores data in a local `.db` file. No Docker or external database is required!

<a id="api-keys"></a>
### 0.2) API Keys

For this notebook, you will need an NVIDIA API key for:
- Embeddings (to convert text to vectors)
- Reranking (to improve search relevance)
- LLM inference (for generating responses)

You can obtain an NVIDIA Build API Key by:
1. Creating an [NVIDIA Build](https://build.nvidia.com) account
2. Generating a key at https://build.nvidia.com/settings/api-keys

Run the cell below to set your API key:

In [ ]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key
else:
    print("NVIDIA_API_KEY is already set.")

<a id="milvus-setup"></a>
### 0.3) Vector Database (Milvus Lite)

This tutorial uses **Milvus Lite**, a lightweight in-process vector database that stores data in a local `.db` file. This eliminates the need for Docker or external database services.

**Advantages of Milvus Lite:**
- No Docker or external services required
- Data persists in a local file (`./milvus.db`)
- Perfect for development, testing, and tutorials
- Same API as full Milvus for easy migration to production

We will define our database path as a variable for use throughout the notebook:

In [ ]:
# Define the Milvus Lite database path
import os

MILVUS_DB_PATH = "./milvus.db"
COLLECTION_NAME = "rag_tutorial_docs"
EMBEDDING_DIMENSION = 1024  # NV-Embed-QA model dimension

print(f"Milvus database will be stored at: {os.path.abspath(MILVUS_DB_PATH)}")
print(f"Collection name: {COLLECTION_NAME}")

Install Milvus Lite (pymilvus includes it by default):

In [ ]:
# Install pymilvus (includes Milvus Lite)
%pip install -q pymilvus

# Test Milvus Lite connection
from pymilvus import MilvusClient

def test_milvus_lite_connection():
    """Test that Milvus Lite is working correctly."""
    try:
        # Create a temporary client to verify installation
        test_client = MilvusClient(uri=MILVUS_DB_PATH)
        collections = test_client.list_collections()
        print(f"✓ Milvus Lite is working!")
        print(f"  Database path: {MILVUS_DB_PATH}")
        print(f"  Existing collections: {collections if collections else 'None'}")
        return True
    except Exception as e:
        print(f"✗ Error initializing Milvus Lite: {e}")
        return False

test_milvus_lite_connection()

<a id="installing-plugin"></a>
### 0.4) Installing the RAG Plugin

Install the NeMo Agent toolkit and the RAG plugin using `uv` for faster dependency resolution.

In [ ]:
%pip install uv

In [ ]:
%%bash
# Check if nvidia-nat is installed, if not install it
uv pip show -q "nvidia-nat"
if [ $? -ne 0 ]; then
    uv pip install "nvidia-nat[langchain]"
else
    echo "nvidia-nat is already installed"
fi

In [ ]:
%%bash
# Install the RAG plugin
# If you're in the NeMo-Agent-Toolkit repository, install from local:
# uv pip install -e packages/nvidia_nat_rag

# Or install from PyPI (when available):
# uv pip install nvidia-nat-rag

# For this notebook, we assume you're in the repository:
cd ../../.. && uv pip install -e packages/nvidia_nat_rag

Verify the plugin is installed correctly:

In [ ]:
# Verify the plugin functions are available
!nat info functions | grep -E "nvidia_rag_query|nvidia_rag_search"

<a id="custom-ingestion"></a>
## 1) Custom Document Ingestion

Before we can query documents, we need to ingest them into our Milvus Lite database. This section demonstrates how to:
1. Create sample documents
2. Generate embeddings using NVIDIA's embedding model
3. Create a collection and insert the documents
4. Verify the ingestion

This is similar to the ingestion process shown in the `ingestion_api_usage.ipynb` notebook, but uses Milvus Lite directly instead of a server-based approach.

<a id="create-sample-docs"></a>
### 1.1) Create Sample Documents

Let's create some sample documents to ingest. In a real scenario, you would load documents from files (PDF, DOCX, etc.) and chunk them appropriately.

In [ ]:
# Sample documents for the tutorial
# Each document has content and metadata

SAMPLE_DOCUMENTS = [
    {
        "id": 1,
        "document_name": "product_catalog.pdf",
        "document_type": "pdf",
        "content": """NVIDIA H100 GPU Specifications:
        - 80GB HBM3 memory
        - 3.35TB/s memory bandwidth
        - 4th generation Tensor Cores
        - Transformer Engine for AI training
        - Price: Starting at $30,000 USD
        - Power consumption: 700W TDP""",
        "metadata": {"category": "hardware", "timestamp": "2024-01-15T10:00:00"}
    },
    {
        "id": 2,
        "document_name": "product_catalog.pdf",
        "document_type": "pdf",
        "content": """NVIDIA A100 GPU Specifications:
        - 80GB HBM2e memory
        - 2TB/s memory bandwidth
        - 3rd generation Tensor Cores
        - Multi-Instance GPU (MIG) support
        - Price: Starting at $15,000 USD
        - Power consumption: 400W TDP""",
        "metadata": {"category": "hardware", "timestamp": "2024-01-15T10:00:00"}
    },
    {
        "id": 3,
        "document_name": "support_policy.pdf",
        "document_type": "pdf",
        "content": """Customer Support Policy:
        - 24/7 technical support for enterprise customers
        - Response time: 4 hours for critical issues
        - Software updates included for 3 years
        - On-site support available for additional fee
        - Contact: support@nvidia.com""",
        "metadata": {"category": "support", "timestamp": "2024-02-01T09:00:00"}
    },
    {
        "id": 4,
        "document_name": "shipping_info.pdf",
        "document_type": "pdf",
        "content": """Shipping Information:
        - Free shipping on orders over $1,000
        - Standard delivery: 5-7 business days
        - Express delivery: 2-3 business days ($50 extra)
        - International shipping available
        - Tracking provided for all orders""",
        "metadata": {"category": "logistics", "timestamp": "2024-02-10T14:30:00"}
    },
    {
        "id": 5,
        "document_name": "return_policy.pdf",
        "document_type": "pdf",
        "content": """Return Policy:
        - 30-day return window for unopened products
        - 14-day return window for opened products
        - Full refund for defective items
        - RMA required for all returns
        - Restocking fee: 15% for opened products""",
        "metadata": {"category": "policy", "timestamp": "2024-01-20T11:00:00"}
    },
]

print(f"Created {len(SAMPLE_DOCUMENTS)} sample documents:")
for doc in SAMPLE_DOCUMENTS:
    print(f"  - {doc['document_name']}: {doc['content'][:50]}...")

<a id="generate-embeddings"></a>
### 1.2) Generate Embeddings

We'll use NVIDIA's NV-Embed-QA model to generate embeddings for our documents. This model is optimized for retrieval tasks and produces 1024-dimensional vectors.

In [ ]:
# Install required packages for embeddings
%pip install -q langchain-nvidia-ai-endpoints

In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

# Initialize the NVIDIA embeddings model
embeddings_model = NVIDIAEmbeddings(
    model="nvidia/nv-embedqa-e5-v5",
    truncate="END",
)

# Generate embeddings for all documents
print("Generating embeddings for documents...")
document_texts = [doc["content"] for doc in SAMPLE_DOCUMENTS]
embeddings = embeddings_model.embed_documents(document_texts)

print(f"✓ Generated {len(embeddings)} embeddings")
print(f"  Embedding dimension: {len(embeddings[0])}")

# Update EMBEDDING_DIMENSION based on actual model output
EMBEDDING_DIMENSION = len(embeddings[0])
print(f"  Updated EMBEDDING_DIMENSION to: {EMBEDDING_DIMENSION}")

<a id="ingest-documents"></a>
### 1.3) Create Collection and Ingest Documents

Now we'll create a Milvus collection and insert our documents with their embeddings. The collection schema includes:
- `id`: Unique document identifier
- `vector`: The embedding vector
- `text`: The document content (used for retrieval)
- `document_name`: Source file name
- `document_type`: File type (pdf, docx, etc.)
- `category`: Document category for filtering

In [ ]:
from pymilvus import MilvusClient, DataType

def create_collection_and_ingest(
    db_path: str,
    collection_name: str,
    documents: list,
    embeddings: list,
    embedding_dim: int
):
    """Create a Milvus collection and ingest documents with embeddings."""
    
    # Initialize Milvus Lite client
    client = MilvusClient(uri=db_path)
    
    # Drop collection if it exists (for clean re-runs)
    if client.has_collection(collection_name):
        print(f"Dropping existing collection: {collection_name}")
        client.drop_collection(collection_name)
    
    # Create collection schema
    schema = client.create_schema(auto_id=False, enable_dynamic_field=True)
    
    # Add fields to schema
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=embedding_dim)
    schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=65535)
    schema.add_field(field_name="document_name", datatype=DataType.VARCHAR, max_length=512)
    schema.add_field(field_name="document_type", datatype=DataType.VARCHAR, max_length=64)
    schema.add_field(field_name="category", datatype=DataType.VARCHAR, max_length=128)
    
    # Create index parameters
    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name="vector",
        index_type="FLAT",  # Use FLAT for small datasets, IVF_FLAT for larger
        metric_type="COSINE",
        params={}
    )
    
    # Create the collection
    print(f"Creating collection: {collection_name}")
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params
    )
    
    # Prepare data for insertion
    data = []
    for doc, embedding in zip(documents, embeddings):
        data.append({
            "id": doc["id"],
            "vector": embedding,
            "text": doc["content"],
            "document_name": doc["document_name"],
            "document_type": doc["document_type"],
            "category": doc["metadata"]["category"]
        })
    
    # Insert data
    print(f"Inserting {len(data)} documents...")
    result = client.insert(collection_name=collection_name, data=data)
    
    print(f"✓ Successfully ingested {result['insert_count']} documents")
    return client

# Run the ingestion
milvus_client = create_collection_and_ingest(
    db_path=MILVUS_DB_PATH,
    collection_name=COLLECTION_NAME,
    documents=SAMPLE_DOCUMENTS,
    embeddings=embeddings,
    embedding_dim=EMBEDDING_DIMENSION
)

<a id="verify-ingestion"></a>
### 1.4) Verify Ingestion

Let's verify that our documents were successfully ingested by querying the collection and testing a semantic search.

In [ ]:
# Verify ingestion by listing collections and checking document count
def verify_ingestion(client: MilvusClient, collection_name: str):
    """Verify the ingestion was successful."""
    
    # List all collections
    collections = client.list_collections()
    print(f"Collections in database: {collections}")
    
    # Get collection stats
    stats = client.get_collection_stats(collection_name)
    print(f"\nCollection '{collection_name}' stats:")
    print(f"  Row count: {stats['row_count']}")
    
    # Query all documents
    results = client.query(
        collection_name=collection_name,
        filter="",
        output_fields=["id", "document_name", "category"],
        limit=10
    )
    
    print(f"\nDocuments in collection:")
    for doc in results:
        print(f"  ID {doc['id']}: {doc['document_name']} (category: {doc['category']})")
    
    return len(results)

verify_ingestion(milvus_client, COLLECTION_NAME)

In [ ]:
# Test semantic search on the ingested documents
def test_semantic_search(
    client: MilvusClient,
    collection_name: str,
    query: str,
    embeddings_model,
    top_k: int = 3
):
    """Test semantic search on the collection."""
    
    print(f"Query: '{query}'")
    print("-" * 50)
    
    # Generate embedding for the query
    query_embedding = embeddings_model.embed_query(query)
    
    # Search the collection
    results = client.search(
        collection_name=collection_name,
        data=[query_embedding],
        limit=top_k,
        output_fields=["text", "document_name", "category"]
    )
    
    print(f"\nTop {top_k} results:")
    for i, hit in enumerate(results[0]):
        print(f"\n{i+1}. Score: {hit['distance']:.4f}")
        print(f"   Document: {hit['entity']['document_name']}")
        print(f"   Category: {hit['entity']['category']}")
        print(f"   Content: {hit['entity']['text'][:150]}...")

# Test a few queries
test_semantic_search(milvus_client, COLLECTION_NAME, "What is the price of the H100 GPU?", embeddings_model)
print("\n" + "=" * 70 + "\n")
test_semantic_search(milvus_client, COLLECTION_NAME, "How can I return a product?", embeddings_model)

<a id="understanding-functions"></a>
## 2) Understanding the RAG Functions

The RAG plugin provides two main functions that can be used as tools in your agent workflows:

<a id="rag-query"></a>
### 2.1) nvidia_rag_query - Get AI-Generated Responses

The `nvidia_rag_query` function retrieves relevant documents and uses an LLM to generate a response based on them.

**Configuration Options:**
| Parameter | Type | Description | Default |
|-----------|------|-------------|---------|
| `config_file` | string | Path to NVIDIA RAG config YAML | `config.yaml` |
| `collection_names` | list[str] | Milvus collection names to query | `[]` |
| `vdb_endpoint` | string | Vector database endpoint URL or local db path | `http://localhost:19530` or `./milvus.db` |
| `embedding_endpoint` | string | Custom embedding endpoint (optional) | `None` |
| `use_knowledge_base` | bool | Use knowledge base for RAG | `true` |

<a id="rag-search"></a>
### 2.2) nvidia_rag_search - Search Document Chunks

The `nvidia_rag_search` function performs semantic search and returns relevant document chunks without generating a response.

**Configuration Options:**
| Parameter | Type | Description | Default |
|-----------|------|-------------|---------|
| `config_file` | string | Path to NVIDIA RAG config YAML | `config.yaml` |
| `collection_names` | list[str] | Milvus collection names to search | `[]` |
| `vdb_endpoint` | string | Vector database endpoint URL or local db path | `http://localhost:19530` or `./milvus.db` |
| `embedding_endpoint` | string | Custom embedding endpoint (optional) | `None` |
| `reranker_top_k` | int | Results after reranking | `10` |
| `vdb_top_k` | int | Results from vector search | `100` |

<a id="cli-usage"></a>
## 3) Using the RAG Plugin via CLI

The simplest way to use the RAG plugin is through the `nat run` command.

<a id="cli-config"></a>
### 3.1) Creating a Workflow Configuration

First, create a workflow configuration file that defines your RAG tools and agent. Note that we're using our local Milvus Lite database (`./milvus.db`) and the collection we created earlier:

In [ ]:
%%writefile rag_workflow_config.yml
# NVIDIA RAG Workflow Configuration
# This configuration sets up a ReAct agent with RAG capabilities
# Using Milvus Lite with local database file

functions:
  # RAG Query Tool - Returns AI-generated responses from documents
  rag_query:
    _type: nvidia_rag_query
    config_file: config.yaml              # Path to nvidia_rag config
    collection_names: ["rag_tutorial_docs"] # Collection we created earlier
    vdb_endpoint: "./milvus.db"           # Local Milvus Lite database
    use_knowledge_base: true

  # RAG Search Tool - Returns relevant document chunks
  rag_search:
    _type: nvidia_rag_search
    config_file: config.yaml
    collection_names: ["rag_tutorial_docs"]
    vdb_endpoint: "./milvus.db"           # Local Milvus Lite database
    reranker_top_k: 3                      # Reduce for shorter responses
    vdb_top_k: 20

  # Utility tool for date/time queries
  current_datetime:
    _type: current_datetime

llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.0

workflow:
  _type: react_agent
  tool_names:
    - rag_query
    - rag_search
    - current_datetime
  llm_name: nim_llm
  verbose: true
  parse_agent_response_max_retries: 3
  description: "An assistant that answers questions using RAG over your documents"

You also need to create or reference an NVIDIA RAG configuration file (`config.yaml`). Here's an example:

In [ ]:
%%writefile config.yaml
# NVIDIA RAG Configuration
# See: https://github.com/NVIDIA-AI-Blueprints/rag
# Using Milvus Lite with local database file

embedder:
  model_name: nvidia/nv-embedqa-e5-v5
  # For on-premises deployment, specify endpoint:
  # endpoint: http://localhost:9080

llm:
  model_name: meta/llama-3.1-70b-instruct
  temperature: 0.0
  max_tokens: 1024

reranker:
  model_name: nvidia/nv-rerankqa-mistral-4b-v3
  # For on-premises deployment, specify endpoint:
  # endpoint: http://localhost:9081

vector_store:
  # Using Milvus Lite with local database file
  endpoint: ./milvus.db

<a id="cli-run"></a>
### 3.2) Running with nat run Command

Use the `nat run` command to execute the workflow:

In [ ]:
# Run the RAG workflow with a query
!nat run --config_file rag_workflow_config.yml --input "What information do you have about product pricing?"

You can also run queries interactively:

In [ ]:
# Example: Search for documents without generating a response
!nat run --config_file rag_workflow_config.yml --input "Search for documents about customer support"

<a id="python-api"></a>
## 4) Using the RAG Plugin Programmatically

For more control and integration into Python applications, you can use the Python API.

<a id="run-workflow"></a>
### 4.1) Using run_workflow Function

The simplest programmatic approach is using the `run_workflow` function:

In [ ]:
import asyncio
from nat.utils import run_workflow

async def query_rag_workflow(query: str) -> str:
    """Run a RAG query using the workflow system."""
    result = await run_workflow(
        config_file='rag_workflow_config.yml',
        prompt=query
    )
    return result

# Run the query
result = asyncio.run(query_rag_workflow("What products do you have information about?"))
print(result)

You can also run multiple queries in sequence using the same workflow:

In [ ]:
async def run_multiple_queries():
    """Run multiple RAG queries."""
    queries = [
        "What is the price of product A?",
        "Summarize the main features of product B",
        "What are the shipping options?"
    ]
    
    results = []
    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        result = await run_workflow(
            config_file='rag_workflow_config.yml',
            prompt=query
        )
        results.append(result)
        print(f"Answer: {result[:200]}..." if len(result) > 200 else f"Answer: {result}")
    
    return results

# Uncomment to run
# results = asyncio.run(run_multiple_queries())

<a id="workflow-builder"></a>
### 4.2) Using WorkflowBuilder Directly

For more advanced use cases, you can use the `WorkflowBuilder` directly. This gives you access to sessions, streaming, and more control:

In [ ]:
from nat.builder.workflow_builder import WorkflowBuilder
from nat.runtime.loader import load_config
from nat.runtime.session import SessionManager

async def advanced_rag_workflow():
    """Demonstrate advanced workflow usage with session management."""
    
    # Load the configuration
    config = load_config('rag_workflow_config.yml')
    
    # Create the workflow builder
    async with WorkflowBuilder.from_config(config=config) as workflow_builder:
        # Create a session manager
        session_manager = await SessionManager.create(
            config=config, 
            shared_builder=workflow_builder
        )
        
        try:
            # Create a session and run queries
            async with session_manager.session() as session:
                # First query
                async with session.run("What documents are available?") as runner:
                    result1 = await runner.result(to_type=str)
                    print(f"Query 1 Result: {result1}")
                
                # Second query (maintains conversation context)
                async with session.run("Can you tell me more about the first one?") as runner:
                    result2 = await runner.result(to_type=str)
                    print(f"Query 2 Result: {result2}")
        finally:
            await session_manager.shutdown()

# Uncomment to run
# asyncio.run(advanced_rag_workflow())

### 4.3) Programmatic Configuration (Without YAML)

You can also create configurations programmatically without YAML files. Note that we use the local Milvus Lite database:

In [ ]:
from nat.data_models.config import Config

# Define configuration as a Python dictionary
# Using local Milvus Lite database
config_dict = {
    "functions": {
        "rag_query": {
            "_type": "nvidia_rag_query",
            "config_file": "config.yaml",
            "collection_names": ["rag_tutorial_docs"],  # Our ingested collection
            "vdb_endpoint": "./milvus.db",              # Local Milvus Lite
            "use_knowledge_base": True
        },
        "current_datetime": {
            "_type": "current_datetime"
        }
    },
    "llms": {
        "nim_llm": {
            "_type": "nim",
            "model_name": "meta/llama-3.1-70b-instruct",
            "temperature": 0.0
        }
    },
    "workflow": {
        "_type": "react_agent",
        "tool_names": ["rag_query", "current_datetime"],
        "llm_name": "nim_llm",
        "verbose": True
    }
}

# Create config object
config = Config.model_validate(config_dict)

async def run_with_programmatic_config():
    result = await run_workflow(
        config=config,
        prompt="What information is available in the knowledge base?"
    )
    return result

# Uncomment to run
# result = asyncio.run(run_with_programmatic_config())
# print(result)

<a id="direct-rag"></a>
## 5) Using NVIDIA RAG Client Directly

For scenarios where you don't need the full workflow system (such as no agent reasoning, no tool selection), you can use the NVIDIA RAG client directly.

<a id="without-workflow"></a>
### 5.1) Without the Workflow System

The NVIDIA RAG library can be used independently for direct document queries:

In [ ]:
import json
from nvidia_rag import NvidiaRAG
from nvidia_rag.utils.configuration import NvidiaRAGConfig

async def direct_rag_query(query: str, collection_names: list[str]):
    """
    Query documents using NVIDIA RAG directly without the workflow system.
    
    This is useful when you:
    - Don't need agent reasoning or tool selection
    - Want direct control over the RAG pipeline
    - Are building a simple Q&A interface
    """
    # Initialize RAG client
    rag_config = NvidiaRAGConfig.from_yaml("config.yaml")
    rag = NvidiaRAG(config=rag_config)
    
    # Send the query
    response = await rag.generate(
        messages=[{"role": "user", "content": query}],
        use_knowledge_base=True,
        collection_names=collection_names,
    )
    
    if response.status_code != 200:
        return f"Error: RAG query failed with status code {response.status_code}"
    
    # Extract the response from the streaming generator
    full_response = []
    async for chunk in response.generator:
        if chunk.startswith("data: "):
            chunk = chunk[len("data: "):].strip()
        if not chunk:
            continue
        try:
            data = json.loads(chunk)
            choices = data.get("choices", [])
            if choices:
                delta = choices[0].get("delta", {})
                text = delta.get("content")
                if not text:
                    message = choices[0].get("message", {})
                    text = message.get("content", "")
                if text:
                    full_response.append(text)
        except json.JSONDecodeError:
            continue
    
    return "".join(full_response) if full_response else "No response generated."

# Example usage with our ingested documents
# result = asyncio.run(direct_rag_query(
#     query="What is the return policy?",
#     collection_names=["rag_tutorial_docs"]
# ))
# print(result)

### 5.2) Direct Document Search

You can also search for documents without generating a response:

In [ ]:
async def direct_rag_search(query: str, collection_names: list[str], top_k: int = 5):
    """
    Search for relevant documents using NVIDIA RAG directly.
    
    Returns document chunks without generating an LLM response.
    Useful for:
    - Building custom retrieval pipelines
    - Inspecting what documents are retrieved
    - Debugging retrieval quality
    """
    # Initialize RAG client
    rag_config = NvidiaRAGConfig.from_yaml("config.yaml")
    rag = NvidiaRAG(config=rag_config)
    
    # Search for documents
    citations = await rag.search(
        query=query,
        collection_names=collection_names,
        reranker_top_k=top_k,
        vdb_top_k=top_k * 4,  # Search more, then rerank
    )
    
    if not citations or not hasattr(citations, "results") or not citations.results:
        return "No documents found for the given query."
    
    # Format results
    results = []
    for idx, citation in enumerate(citations.results):
        doc_name = getattr(citation, "document_name", f"Document {idx + 1}")
        content = getattr(citation, "content", "")
        doc_type = getattr(citation, "document_type", "text")
        
        results.append({
            "document_name": doc_name,
            "document_type": doc_type,
            "content": content[:500] + "..." if len(content) > 500 else content
        })
    
    return results

# Example usage with our ingested documents
# results = asyncio.run(direct_rag_search(
#     query="product specifications",
#     collection_names=["rag_tutorial_docs"],
#     top_k=3
# ))
# for result in results:
#     print(f"\n{result['document_name']} ({result['document_type']})")
#     print(f"{result['content']}")

### 5.3) Comparison: Workflow vs Direct Usage

| Feature | Workflow System | Direct RAG Client |
|---------|-----------------|-------------------|
| Agent reasoning | Yes | No |
| Tool selection | Yes | No |
| Multiple tools | Yes | Single function |
| Session management | Yes | Manual |
| Configuration | YAML or Python dict | Python only |
| Best for | Complex workflows | Simple Q&A |

<a id="troubleshooting"></a>
## 6) Troubleshooting

### Common Issues and Solutions

### Issue: Function type `nvidia_rag_query` not found

**Solution:** The RAG plugin is not installed. Install it with:
```bash
uv pip install -e packages/nvidia_nat_rag
```

### Issue: Milvus Lite Database Issues

**Solution:** Verify the database file exists and is accessible:

In [ ]:
# Check Milvus Lite database file
import os
if os.path.exists(MILVUS_DB_PATH):
    size_mb = os.path.getsize(MILVUS_DB_PATH) / (1024 * 1024)
    print(f"✓ Milvus Lite database exists: {MILVUS_DB_PATH}")
    print(f"  Size: {size_mb:.2f} MB")
else:
    print(f"✗ Milvus Lite database not found at: {MILVUS_DB_PATH}")
    print("  Run the ingestion cells above to create the database.")

### Issue: Token limit exceeded

**Solution:** Reduce the number of results returned:
```yaml
rag_search:
  _type: nvidia_rag_search
  reranker_top_k: 1    # Reduce from default
  vdb_top_k: 10        # Reduce from default
```

### Issue: NVIDIA API key not set

**Solution:** Set the environment variable:

In [ ]:
# Verify API key is set
if "NVIDIA_API_KEY" in os.environ:
    print("NVIDIA_API_KEY is set")
    print(f"Key starts with: {os.environ['NVIDIA_API_KEY'][:10]}...")
else:
    print("NVIDIA_API_KEY is NOT set!")
    print("Run: export NVIDIA_API_KEY='your-key-here'")

### Issue: No documents found

**Solution:** Verify that:
1. Documents have been ingested into Milvus Lite (run Section 1 of this notebook)
2. Collection names in config match your Milvus collections (`rag_tutorial_docs`)
3. The query is relevant to your document content

In [ ]:
# List Milvus Lite collections
try:
    from pymilvus import MilvusClient
    
    client = MilvusClient(uri=MILVUS_DB_PATH)
    collections = client.list_collections()
    print(f"Available collections: {collections}")
    
    for collection in collections:
        stats = client.get_collection_stats(collection)
        print(f"  - {collection}: {stats['row_count']} documents")
except ImportError:
    print("Install pymilvus: pip install pymilvus")
except Exception as e:
    print(f"Could not read Milvus Lite database: {e}")

## Summary

In this notebook, we covered:

1. **Custom Document Ingestion**: How to create embeddings and ingest documents into Milvus Lite
2. **RAG Functions**: Understanding `nvidia_rag_query` and `nvidia_rag_search`
3. **CLI Usage** (`nat run`): Best for quick queries and testing
4. **Python API** (`run_workflow`): Best for integration into Python applications
5. **Direct RAG Client**: Best for simple Q&A without agent reasoning

All examples use **Milvus Lite** with a local `.db` file, eliminating the need for Docker or external database services. Choose the approach that best fits your use case!

## Cleanup

Remove temporary files created during this notebook. You can optionally keep the `milvus.db` file to preserve your ingested documents:

In [ ]:
# Clean up temporary config files
for f in ["rag_workflow_config.yml", "config.yaml"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

# Optionally remove the Milvus Lite database
# Uncomment the following lines to delete the database:
# if os.path.exists(MILVUS_DB_PATH):
#     os.remove(MILVUS_DB_PATH)
#     print(f"Removed {MILVUS_DB_PATH}")